# Fluxy example

This notebook shows you basic usage of Fluxy using the test dataset.

### Input data

- Flux and concentration netCDF files with model outputs
- json files with models, sites and species information


Please refer to the User Guide for information on files format, filename convention and folder structure.

### Notebook setup

This notebook is designed to be run as it is. 
However you are free to adapt it to your needs.

Here are some things you could try to change:

* Decide the verbosity level by assigning the logging level to `logging.INFO` or `logging.WARNING`.

* Edit the `data_dir` to point towards where the model output is.

* Set `presentation_mode` to `True`/`False` to use big font sizes (recommended for slides) or small font sizes (recommended for text documents). 

* Update the `models` list to select the models you want to plot.


### Import and configuration

There are various configurations that fluxy requires. 
This first cell load most of the necessary one.

In [ ]:
import logging
from fluxy.config import set_print_settings
from fluxy.io import read_config_files
from fluxy.test_utils import data_dir
from fluxy.config import set_model_colors
from fluxy.config import set_model_labels

# Logging settings, fluxy uses the standard logging module
logging.basicConfig(level=logging.WARNING)


# Path to test files
data_dir = data_dir


# Set presentation_mode to True for bigger fonts
presentation_mode = False

# If False, uses default labels. If True, uses labels in models_info.json.
get_labels_from_file = False


# Group the models of interest in meaningful experiment names
models = [
    "InTEM_NAME_EUROPE_EDGAR_old_format",
    "ELRIS_NAME_EUROPE_EDGAR",
    "RHIME_NAME_EUROPE_EDGAR_old_format",
]

# This is the substance available in the test data.
species = "hfc134a" 

# Read the default configuration files
config_data = read_config_files()

annotate_coords = set_print_settings(presentation_mode)
model_colors = set_model_colors(models)
model_labels = set_model_labels(models, config_data, get_labels_from_file)

## 1. Timeseries of country/region fluxes

#### Loading the data

Here we load the data using the `read_model_output` function.  

It will try to look for the files you require in the `data_dir` folder, and will return a dictionary with the model names as keys and the data as values.

Then we slice the data to select only the time period and regions of interest
using the `slice_flux` function.



In [ ]:
from fluxy.io import read_model_output
from fluxy.operators.select import slice_flux


###################################
### edit variables in this block
regions = ["GERMANY", "UK", "BENELUX", "NW_EU2"]
# inversion period, must be a string or a list of the same length as models, e.g. ['monthly','yearly']
period = "yearly"
# desired units for the plot. Add "CO2-eq" to convert the mass to CO2 equivalent (e.g. "Gg CO2-eq yr-1")
country_flux_units_print = "Gg yr-1"
# inclusive
start_date = "2018-01-01"
# not inclusive
end_date = "2024-01-01"


###################################



ds_all_flux = read_model_output(
    data_dir,
    file_type="flux",
    species=species,
    models=models,
    config_data=config_data,
    period=period,
)

# Select only the time period and regions of interest
ds_all_flux_scaled = slice_flux(
    ds_all_flux,
    config_data,
    start_date,
    end_date,
    species=species,
    country_flux_units_print=country_flux_units_print,
)



#### Look at one dataset 

Fluxy stores all the data in a dictionary of xarray datasets. One dataset for each model.

Below we show an example of how the data is stored internally.
Note that you don't need to understand much about xarray to use Fluxy.

In [ ]:
ds_all_flux_scaled.keys()

In [ ]:

ds_all_flux_scaled[models[0]]

#### Plot timeseries of country fluxes

In [ ]:
from fluxy.plots.flux_timeseries import plot_country_flux

###################################
### edit variables in this block
plot_inventory = False
inventory_years = (
    None  # If None, plots most recent. Or can choose list of years: ['2022','2023']
)
inventory_filename = "UNFCCC_inventory"  # Full filename = {inventory_filename}_{species}_{inventory_year}
fix_y_axes = False  # if True: all y axis limits are the same, if False: each y axis is relative to the data
# if a list of floats (e.g. [0,0.1]) applies these limit to all axes
add_prior = True  # if True: plots prior as dashed lines
add_prior_unc = False  # if True: plots prior uncertainty as shaded area
set_global_leg = (
    True  # If True, plots one single legend instead of one legend per subplot.
)
country_codes_as_titles = False  # If True, lists 3-letter country codes under region names in subplot titles. Set to None for no title.
plot_separate = True  # If True, includes all model results as separate lines (or insert a list of boolean of the same length as models to specify which models to plot)
plot_combined = False  # If True, combined results, averaged from all models (or insert a list of boolean of the same length as models to specify which models to combine)
resample = None  # If None, no resample is done. Else resample the data to the given period (options 'year' and 'season' for yearly and seasonal averages)
resample_uncert_correlation = False  # If True, uses mean uncertainty during resampling, if False, recalculates uncertainty assuming no correlation.
plot_resample_and_original = (
    False  # If True, plots both the resampled and original data
)
annex_mode = (
    False  # If True, replace the labels with more concise versions for NID Annexes.
)
rolling_mean = False  ##If True, calculates a rolling mean of the data (insert a list of boolean of the same length as models to specify the models to smooth)
###################################

fig = plot_country_flux(
    ds_all_flux_scaled,
    species,
    plot_regions=regions,
    config_data=config_data,
    model_colors=model_colors,
    model_labels=model_labels,
    start_date=start_date,
    end_date=end_date,
    annex_mode=annex_mode,
    plot_inventory=plot_inventory,
    inventory_years=inventory_years,
    inventory_filename=inventory_filename,
    data_dir=data_dir,
    fix_y_axes=fix_y_axes,
    add_prior=add_prior,
    add_prior_unc=add_prior_unc,
    set_global_leg=set_global_leg,
    country_codes_as_titles=country_codes_as_titles,
    plot_separate=plot_separate,
    plot_combined=plot_combined,
    resample=resample,
    resample_uncert_correlation=resample_uncert_correlation,
    plot_resample_and_original=plot_resample_and_original,
    rolling_mean=rolling_mean,
)

If you know matplotlib, you will see that you can customize the plot further as you wish,
because fluxy returns the matplotlib figure object.

For example here, we save the plot

In [ ]:
output_path = data_dir / 'flux_country.png'

fig.savefig(output_path,bbox_inches='tight',pad_inches=0.2,dpi=300)

## 2. Modelled and observed mole fractions and/or baselines

### Read and clean the data

Now we want to load timeseries of mole fraction data containing the comparisons between 
model and observations.

In [ ]:
from fluxy.io import read_model_output
from fluxy.operators.select import slice_mf

###################################
site = "MHD"
# inversion period, must be a string or a list of the same length as models, e.g. ['monthly','yearly']
period = "yearly"
mf_units_print = "ppt"
start_date = "2018-01-01"
end_date = "2019-01-01"
#'MHD', 'JFJ' or 'CMN'. If None, does not mask by baseline time
baseline_site = None
baseline_filename = "InTEM_baseline_timestamps"
###################################

ds_all_mf = read_model_output(
    data_dir,
    file_type="concentration",
    species=species,
    models=models,
    config_data=config_data,
    period=period,
)

ds_all_mf_sliced = slice_mf(
    ds_all_mf.copy(),
    start_date,
    end_date,
    site,
    baseline_site=baseline_site,
    baseline_filename=baseline_filename,
    data_dir=data_dir,
    mf_units_print=mf_units_print,
)


### Plot timestamps with data at each site

This plot shows you the data availability at each site in time.

In [ ]:
from fluxy.plots.mf_timeseries import plot_sites_timeseries

fig = plot_sites_timeseries(
    ds_all_mf,
    "mf_posterior",
    start_date,
    end_date,
    model_colors,
    model_labels,
    margin=0.2,
)

### Timeseries plot, separated by model

This shows timeseries of each model, with the data distribution.

In [ ]:
from fluxy.plots.mf_timeseries import plot_mf_timeseries

###################################
### edit variables in this block
# Variables and respective uncertainties to plot
include = {"mf_observed": None, "mf_posterior": "percentile_mf_posterior"}

# To plot the histogram of the variables in "include", set "diff_include" to None
# To plot the histogram of Obs-variable, set "diff_include" to the desired variable to be subtracted
diff_include = ["mf_posterior"]

# To choose y-axis limits set y_lim=[min_value,max_value]
y_lim = None
###################################

###################################
### options for variables to include
# mf_observed         - total observed mole fraction
# mf_prior            - prior total mole fraction
# mf_posterior        - posterior total mole fraction
# mf_bc_prior         - prior baseline
# mf_bc_posterior     - posterior baseline
# mf_bias_prior       - prior bias added to site
# mf_bias_posterior   - posterior bias added to site
# mf_outer_prior      - prior mole fractions only from outer regions
# mf_outer_posterior  - posterior mole fractions only from outer regions
# stdev_mf_observed_repeatability - observed repeatability mole fraction uncertainty
# stdev_mf_observed_variability   - observed variability mole fraction uncertainty
# stdev_mf_model                  - model mole fraction uncertainty
# stdev_mf_total                  - total mole fraction uncertainty

### options for uncertainties to plot as error bars/shaded area
# stdev_mf_observed_repeatability - observed repeatability mole fraction uncertainty
# stdev_mf_observed_variability   - observed variability mole fraction uncertainty
# stdev_mf_model                  - model mole fraction uncertainty
# stdev_mf_total                  - total mole fraction uncertainty
# percentile_mf_prior             - prior mole fraction uncertainty
# percentile_mf_posterior         - posterior mole fraction uncertainty
###################################

fig = plot_mf_timeseries(
    ds_all_mf_sliced,
    species,
    site,
    model_colors,
    model_labels,
    config_data,
    annotate_coords,
    presentation_mode,
    plot_type="separate",
    include=include,
    diff_include=diff_include,
    y_lim=y_lim,
)

### Timeseries plot, all models together

Same as before but all the models together in the same plot.

In [ ]:
from fluxy.plots.mf_timeseries import plot_mf_timeseries

###################################
### edit variables in this block (see instructions above)
include = {"mf_posterior": "percentile_mf_posterior"}
diff_include = ["mf_posterior"]
y_lim = None
###################################

fig = plot_mf_timeseries(
    ds_all_mf_sliced,
    species,
    site,
    model_colors,
    model_labels,
    config_data,
    annotate_coords,
    presentation_mode,
    # Option to put all models in the same axis
    plot_type="together",
    include=include,
    diff_include=diff_include,
    y_lim=y_lim,
)

### Absolute differences between two models

This allows you to compare two models.

first we will calulate the difference between two models, and then we will plot the absolute difference.

In [ ]:
from fluxy.operators.mf import compute_mf_difference
from fluxy.plots.mf_timeseries import plot_mf_timeseries

###################################
### edit variables in this block
models_to_subtract = models[:2]  # Models to subtract (must be a list of 2 elements)
include = {"mf_observed": None}  # Variables to plot (dict values must be set to None)
###################################

# Calculate the model difference
ds_diff = compute_mf_difference(ds_all_mf_sliced.copy(), models_to_subtract)

# Call the same mf timeseries plot function
fig = plot_mf_timeseries(
    ds_diff,
    species,
    site,
    model_colors,
    model_labels,
    config_data,
    annotate_coords,
    presentation_mode,
    # Specify the plot type as "diff" to plot the absolute difference
    plot_type="diff",
    include=include,
    diff_include=None,
    y_lim=None,
)

### Statistics

It is also possible to calculate some statistics and plot them using the `plot_stats` function.

In [ ]:
from fluxy.operators.select import slice_mf
from fluxy.operators.mf import stats_mf
from fluxy.plots.mf_stats import plot_stats_mf


###################################
### edit variables in this block
stats_to_plot = ["pearson", "bias", "crmse"]
what_to_compare = "posterior_above_BC"
stats_ylim = {
    "pearson": [0, 1],
    "bias": [-1.5, 0.5],
    "crmse": [0, 1.5],
}  # or set it to None


### Implemented statistics (stats_to_plot)
# pearson - Pearson correlation coefficient
# bias    - Mean difference
# rmse    - RMSE
# nrmse   - Normalised RMSE (division by mean observation)
# crmse   - Centered (bias-corrected) RMSE
# sd_obs  - Standard deviation of observations
# sd_sim  - Standard deviation of simulation
# sd_res  - Standard deviation of residuals
# nn      - Number of observation-simulation pairs
###################################

ds_all_allsites = slice_mf(
    ds_all_mf.copy(),
    start_date,
    end_date,
    site=None,
    baseline_site=baseline_site,
    baseline_filename=baseline_filename,
    data_dir=data_dir,
    mf_units_print=mf_units_print,
)

stats = stats_mf(ds_all_allsites, stats_type=what_to_compare)

fig = plot_stats_mf(
    stats=stats,
    stats_to_plot=stats_to_plot,
    species=species,
    model_colors=model_colors,
    model_labels=model_labels,
    config_data=config_data,
    mf_units_print=mf_units_print,
    stats_type=what_to_compare,
    stats_ylim=stats_ylim,
    start_date=start_date,
    end_date=end_date,
)

### Plot Taylor Diagrams

Taylor diagrams are a great way to visualize the performance of models against observations.

In [ ]:
from fluxy.operators.mf import stats_mf
from fluxy.plots.mf_stats import plot_taylor_diagram

###################################
### edit variables in this block
include = ["prior", "posterior"]  # Variables to include in the Taylor diagram
plot_type_model = "together"  #  If 'separate', plots each model in a separate subplot. If 'together', plots all models in the same subplot.
plot_type_stat = "together"  # If 'separate', plots each statistic in a separate subplot. If 'together', plots all statistics in the same subplot.
normalize = True  # If True, normalizes the data by the standard deviation of the observations. If False, plots the absolute data.
# Marker styles used to plot the stats specified in the `include` parameter, sharing the same index.
stat_markers = ["s"]
# Range for the standard deviation axis, in units provided with `sd_unit` if `normalize` = True.
sd_range = [
    0,
    1.5,
]
check_sites = False  # If True, color different sites with different colors
###################################

stats = {}
for stat in include:
    stats[stat] = stats_mf(ds_all_allsites, stats_type=stat)

fig = plot_taylor_diagram(
    stats=stats,
    model_colors=model_colors,
    model_labels=model_labels,
    stat_markers=stat_markers,
    normalize=normalize,
    plot_type_model=plot_type_model,
    plot_type_stat=plot_type_stat,
    include=include,
    sd_range=sd_range,
    sd_unit=mf_units_print,
    check_sites=check_sites,
)

## 3. Spatial maps of country/region fluxes

Spatial distribution and maps are always a hassle to plot, but Fluxy makes it easy.

### Read the data 

Once again, we use the `read_model_output` function to load the data.

Since the test data is small, the maps will be very coarse.

In [ ]:
from fluxy.io import read_model_output
from fluxy.operators.select import slice_flux

###################################
### edit variables in this block
species = "hfc134a"
start_date = "2018-01-01"
end_date = "2024-01-01"
period = "yearly"  # inversion period, must be a string or a list of the same length as models, e.g. ['monthly','yearly']
flux_units_print = "kg km-2 yr-1"
###################################

ds_all_flux = read_model_output(
    data_dir, "flux", species, models, config_data, period=period
)

ds_all_flux_scaled = slice_flux(
    ds_all_flux,
    config_data,
    start_date,
    end_date,
    species=species,
    flux_units_print=flux_units_print,
)



### Prior and posterior fluxes comparison across models

In [ ]:
from fluxy.plots.flux_map import plot_flux_map

###################################
### edit variables in this block
region = "EUROPE"
add_sites = True
# Add a marker at given locations, options for 'paris', 'london', ... or any value [lon, lat]
add_markers = ["london", "paris"]
season = None  # If specified, plot the seasonal mean (only valable for monthly data), options for 'DJF', 'MAM', 'JJA' and 'SON'
set_fluxlim = "auto"  # Set flux colorbar limits: e.g.[0,10]|'auto' ('auto' = 99th percentile of flux_posterior)
# If True, plot fluxes at inversion's resolution. If False, plot fluxes at prior's resolution.
plot_inversion_grid_flux = True
###################################

fig = plot_flux_map(
    ds_all_flux_scaled,
    species,
    region,
    config_data,
    model_labels,
    season=season,
    add_sites=add_sites,
    add_markers=add_markers,
    set_fluxlim=set_fluxlim,
    plot_inversion_grid_flux=plot_inversion_grid_flux,
)

### Flux comparison between two models

In [ ]:
from fluxy.plots.flux_map import plot_flux_map_model_comparison

###################################
### edit variables in this block
var = "flux_total_posterior_inversion_grid"
models_plot = models[:2]
region = "FRA"
###################################

fig = plot_flux_map_model_comparison(
    ds_all_flux_scaled,
    var,
    models_plot,
    species,
    region,
    config_data,
    model_labels,
    season=season,
    add_sites=add_sites,
    add_markers=add_markers,
    set_fluxlim=set_fluxlim,
)

### Plot prior, posterior or difference fluxes per time interval

In [ ]:
from fluxy.plots.flux_map import plot_flux_map_over_time

region = "FRA"
chop_by = "year"  # Time unit of the averaging period (options for 'year', 'month' and 'season').
# Alternatively, a list of starting dates can be provided (format '2018-01-01') or a list of months (e.g., [[1:3], [7:9]])
dt = 2  # Number of time steps for year and month chop_by options
var = "flux_total_posterior"  # Variable to be plotted. Extra options for 'posterior_prior_diff', 'posterior_mean_diff', 'posterior_prior_diff_inversion_grid', 'posterior_mean_diff_inversion_grid'.
plot_combined = False
add_sites = True
add_markers = ["paris"]
set_fluxlim = "auto"  # Set flux colorbar limits: e.g.[0,10]|'auto' ('auto' = 99th percentile of flux_posterior)
###################################

fig = plot_flux_map_over_time(
    ds_all_flux_scaled,
    var,
    species,
    region,
    config_data,
    model_labels,
    chop_by=chop_by,
    dt=dt,
    plot_combined=plot_combined,
    add_sites=add_sites,
    add_markers=add_markers,
    set_fluxlim=set_fluxlim,
)

## Conclusion

Fluxy is a powerful tool to visualize and analyze flux data or timeseries.

Try now to adapt it to your own data and see how it works!